# **Acne Detection with YOLOV8**


### Import Dependencies


In [ ]:
import os #buat buka file
import matplotlib.pyplot as plt #buat visualisasi
import cv2 #pustaka OpenCV (olah image)

# ini yang penting: Tensorflow
import tensorflow as tf #pustaka buat deep learning
from tensorflow.keras import * #ambil semua isi class nya
from tensorflow.keras.optimizers import Lion
from tensorflow.keras.optimizers import * #ambil semua isi class nya
import keras_cv #nanti YOLOV8 ada di sini

# Atur batch size
BATCH_SIZE = 32 #satu iterasi dia baca 32 image langsung
AUTO = tf.data.AUTOTUNE

### Prepare data

1. Baca data dan labelnya (karena image dan label terpisah)
2. Buat di list baru (isinya path image)

In [8]:
def parse_annotasi_txt(img_path, txt_path):
    '''
    Jadi dalam folder dataset tuh ada 2 folder. images = gambarnya, labels = annotasi koordinat acne dalam list [class_id posisi_atas_x posisi_atas_y lebar_%_dari_image tinggi_%_dari_gambar]
    Kurang lebih prosesnya kek gini:
    1. Buka gambar → tahu ukurannya.
    2. Buka file label → baca per baris (1 baris = 1 koordinat jerawat).
    3. Tiap baris: kalikan persentase dengan ukuran gambar, lalu hitung pojok kotaknya dari titik tengah.
    4. Kembalikan: path gambar, daftar kelas, daftar kotak.
    '''

    # baca gambar
    img = cv2.imread(img_path) #OpenCV tuh cuman bisa baca bentuk scalar, diubah dulu gambarnya ke scalar pake class imread ini
    h = int(img.shape[0]) # akses tinggi gambar
    w = int(img.shape[1]) # akses lebar gambar

    # baca label annotasi
    file_label = open(text_path, 'r')

    # ubah anotasi(koordinat jerawat) ke satu string panjang yang dimasukin ke list
    lines = file_label.read().split('\n')

    #siapkan wadah hasil
    boxes = []
    classes = []

    #kalo baris pertama kosong, kembalikan nilai list yng kosong (gk ada jerawat)
    if lines[0] == '':
        return image_path, classes, boxes
    #kalo gk kosong, iterasi sebanyak jumlah string di list lines
    else:
        for i in range (0, len(lines)):
            obj = lines[i].split(' ') #ambil baris ke i, pecah berdasarkan spasi
            class_ = int(obj[0]) #ubah string pertama (yaitu ID) ke integer

            # ambil 4 angka sisanya (koordinat), ubah ke desimal
            x1 = float(obj[1])
            y1 = float(obj[2])
            w1 = float(obj[3])
            h1 = float(obj[4])

            # konversi ke piksel
            xmin = int((x1*w) - (w1*w)/2.0)
            ymin = int((y1*h) - (h1*h)/2.0)
            xmax = int((x1*w) + (w1*w)/2.0)
            ymax = int((y1*h) + (h1*h)/2.0)

            # simpan hasil
            boxes.append([xmin, ymin, xmax, ymax])
            classes.append(class_)

            # kembalikan hasil
            return img_path, classes, boxes

def create_path_list(path):
    '''
    path masing2 image nanti dibuat list, biar mudah dibaca/diakses
    '''
    full_path = []

    # variabel image nyimpen paths image -> sorted(urutin)
    images = sorted(os.listdir(path))

    # nama path masukin ke list, join dulu sama nama foldernya pake "/" (misal images/foto1.jpg)
    for i in images:
        full_path.append(os.path.join(path, i))
    
    return full_path

# tentukan class
class_ids = ['Acne']
class_mapping = {0: 'Acne'}